In [ ]:
!pip install transformers datasets accelerate evaluate

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"data": "dataset.txt"})
dataset

In [ ]:
dataset = dataset["data"].train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

dataset

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # ← changed from evaluation_strategy
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])

print(f"Perplexity: {perplexity}")

In [ ]:
trainer.save_model("fine_tuned_gpt2_ai_blog")
tokenizer.save_pretrained("fine_tuned_gpt2_ai_blog")

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

'''Fine-tuned GPT-2 Output'''
prompt = "Artificial Intelligence is transforming"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

''' Original GPT-2 (before fine-tuning)'''
base_model = GPT2LMHeadModel.from_pretrained("gpt2")
base_model.to(model.device)

base_output = base_model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    do_sample=True,
)

print("Base GPT-2 Output:\n")
print(tokenizer.decode(base_output[0], skip_special_tokens=True))

In [ ]:
trainer.save_model("final_gpt2_ai_blog_model")
tokenizer.save_pretrained("final_gpt2_ai_blog_model")